# Flet II: Declarative UI

Recall that in the imperative approach we had to tell the framework exactly how to build and update the interface step by step. That is, we manipulate the UI directly by creating controls, changing their properties, and inserting or removing them in response to user actions (or app state). This is still a valid approach especially for small applications.

The problem with the imperative approach is that the state, logic, and building UI elements all live in the same place. Suppose you have $(s_1, \ldots, s_n)$ **state variables**[^state_vars_not_encouraged]. All components that depend on each of these have to respond to changes in the variables. Since the app is build imperatively, the changes in the components in response to the change have to be hard-coded and reasoned individually reasoned with by the developer. This task grows exponentially as $n$ increases. You have seen a glimpse of this in [the previous notebook](/courses/app-dev/01-flet.html#capstone-todo-app). 

From the [Flet blog post](https://flet.dev/blog/introducing-declarative-ui-in-flet):

> Dogfooding Flet — building our own products like the Flet mobile app and the Control Gallery — made it clear that the imperative approach becomes hard to manage as apps grow. That's why **Flet 1.0** introduces a declarative approach alongside the existing imperative API, drawing inspiration from frameworks such as React, SwiftUI, and Jetpack Compose.

[^state_vars_not_encouraged]: Actually, using state variables to ground the application state is not even encouraged natively by the imperative approach. You can just build the application any way you like &mdash; which, incidentally, is also its strength.

## What is declarative UI? 

The declarative approach means you describe [*what*]{.underline} the UI should look like for a given state instead of [*how*]{.underline} to build the UI. In a sense, it's implicit vs explicit programming. As such, the declarative approach will feel like magic and go over your head when you start coding with it (coming from someone with zero frontend experience). The framework figures out the minimal updates needed to reflect the change in UI w.r.t. change in state so it always stays [consistent across renders]{.mark}. 

:::{.callout-note}
We like to write this in the following slogan: $\text{UI} = f(\text{state}).$ Our goal is to make the code simpler, more predictable, and easier to reason about.

:::

## Hello, world! (declarative)

In [5]:
import flet; print(flet.__version__)

0.80.1


Let's reproduce [our previous](/courses/app-dev/01-flet.html#hello-world) "Hello, world!" program using the declarative approach:

```{.python filename="src/hello.py"}
import flet as ft
import asyncio
from random import randint


hello_world = [
    "Hello, world!",
    "¡Hola, mundo!",
    "Bonjour, monde !",
    "Hallo, Welt!",
    "Ciao, mondo!",
    "Olá, mundo!",
    "こんにちは、世界！",
    "안녕하세요, 세계!",
    "你好，世界！",
    "مرحباً، يا عالم!",
]

@ft.component
def Greeting(greeting: str) -> ft.Container:
    return ft.Container(
        ft.Text(greeting, size=60),
        alignment=ft.Alignment.CENTER,
        expand=True
    )

@ft.component
def RollButton(on_click):
    return ft.FloatingActionButton(
        content=ft.Icon(ft.Icons.CASINO, size=60),
        on_click=on_click,
        height=60, width=60
    )


@ft.component
def AppView() -> ft.Column:
    n = len(hello_world)
    greeting, set_greeting = ft.use_state(hello_world[0])
    
    async def roll_greeting(e):
        set_greeting("")
        await asyncio.sleep(0.2)
        set_greeting(hello_world[randint(0, n - 1)])
    
    return ft.Column(
        controls=[
            Greeting(greeting),
            ft.Row(
                controls=[RollButton(roll_greeting)],
                alignment=ft.MainAxisAlignment.END
            ),
        ],
        alignment=ft.Alignment.CENTER,
        expand=True
    )


if __name__ == "__main__":
    ft.run(lambda page: page.render(AppView))
```

<video
  src="./img/flet-declarative/hello-world.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

This behaves exactly as before. The critical difference is this part of the code:

```python
greeting, set_greeting = ft.use_state(hello_world[0])

async def roll_greeting(e):
    set_greeting("")
    await asyncio.sleep(0.2)
    set_greeting(hello_world[randint(0, n - 1)])
```

... and that we're rendering with `ft.render` the function: 

```python
@ft.component
def AppView() -> ft.Control:
    ...
```

which returns a Flet control. This function returns the UI every time the app's state `greeting` changes. The state variable is defined using the `ft.use_state`. We will explain this function shortly, for now think of `greeting` as the state variable that the component monitors for a change, and `set_greeting` (`str -> None`) as a setter. Notice that we aren't making any explicit page updates. The UI automatically re-renders when the value of `greeting` changes.

Also notice that the hook is now asynchronous. This is necessary since the assignment of the value of `use_state` variable is scheduled in the async event loop, along with the UI re-render. For example, if we don't put `asyncio.sleep` here, then the transition into empty text will not register in the UI since control is not [yielded]{.mark} to the Flet framework in between the state variable updates.

## Declarative elements

### Components

A **component** is simply a [reusable function]{.underline} that returns a control as a function of the state. It should be a self-contained unit of UI that takes inputs (properties, data, event handlers) and returns Flet controls. Every time its inputs or internal state change, the component rebuilds its UI, and Flet automatically updates only the changed parts. In the above example:

```python
@ft.component
def Greeting(text: str) -> ft.Container:
    return ft.Container(
        ft.Text(text, size=60),
        alignment=ft.Alignment.CENTER,
        expand=True
    )

@ft.component
def RollButton(on_click):
    return ft.Row(
        controls=[
            ft.FloatingActionButton(
                content=ft.Icon(ft.Icons.CASINO, size=60),
                on_click=on_click,
                height=60, width=60
            )
        ],
        alignment=ft.MainAxisAlignment.END
    )
```

The first one returns a container for a text greeting that is centered, while the other is a row containing the stylized button aligned to the right.

### Hooks, state variables

Hooks are lightweight functions[^hooks_fast] that let components **store state**, **react to lifecycle events**, or **access shared context**. Moreover, these are all accomplished without writing classes or managing manual state objects. In our example, we have `roll_greeting` as a hook that sets the `greeting` variable. Note that when `set_greeting` is called Flet re-runs the component and re-renders only what changed (i.e. the output of the `Greeting` component).

The `use_state` function gives this variable a **persistent state**. That persistence is crucial: ordinary local variables are re-created on every render, so their values would disappear. Hook state survives re-renders, giving your functional components memory, and allowing shared data, without resorting to globals or classes.

:::{.callout-tip}
Flet offers the following built-in hooks: 

| | |
| :------- | :----------------------------------------- |
| [`use_state`](https://docs.flet.dev/types/usestate/) | Store local state across rebuilds. |
| [`use_effect`](https://docs.flet.dev/types/useeffect/) | Run side effects when something changes. | 
| [`use_context`](https://docs.flet.dev/types/usecontext/)  | Access shared data or services. | 
| [`use_memo`](https://docs.flet.dev/types/usememo/) | Memoize computed values. |
| [`use_ref`](https://docs.flet.dev/types/useref/) | Preserve a mutable value for the lifetime of the component without causing re-renders. |

: {tbl-colwidths="[30,70]"}

:::

[^hooks_fast]: It has to be instantaneous for the app to be reactive.

### Observables

One can think of `use_state` variables as **local**, **component-scoped** persistent state. Meanwhile, we have application state that is, in a sense, global and reflects a truth. In the declarative approach, we can represent application state in terms of **observables**. An observable can be thought of as [shallow change signals]{.mark}[^observable_shallow] that allow us to sync UI automatically. That is, whenever the fields of observable change, the corresponding parts of the UI that depend on it are notifed and update instantly and efficiently. 

[^observable_shallow]: Observables are shallow in a sense that it cannot detect deep changes, e.g. in nested observables, or changing attributes of nested Python mutables. This means we have to manually notify the outer observable when changing deeply nested attributes. Or use a flat observable.

Observables is created in Flet preferably by decorating over dataclasses:

```python
@ft.observable
@dataclass
class CounterState:
    count: int
```

Observables fit nicely into Flet's declarative model in that a component that depends on an **observable parameter** automatically re-renders when that observable updates. This is implemented below. Moreover, hooks that reference observables trigger a re-render when the observable changes. For example, we may see examples with:

```python
@ft.component
def AppView() -> ft.Column:
    todo, _ = ft.use_state(AppState())
```

where `AppState` is an observable. This is done in the ff. example:


```{.python filename=src/counter.py}
import asyncio
from dataclasses import dataclass

import flet as ft

@ft.observable
@dataclass
class AppState:
    counter: float

    async def start_counter(self):
        self.counter = 0                # <1>
        for _ in range(0, 10):
            await asyncio.sleep(0.5)
            self.counter += 0.1                 # <2>
            

@ft.component
def AppView():
    state, _ = ft.use_state(AppState(counter=0))    # <3>

    return [
        ft.ProgressBar(state.counter),      # <4>
        ft.Button("Run!", on_click=state.start_counter),    # <5>
    ]

ft.run(lambda page: page.render(AppView))
```

1. Updating an observable to `0` using `=`. Flet detects a change ⇒ re-renders UI. Note that it's important for the sleep to be in between this assignment and the update (i.e. sleep happens before the increment), so that setting to zero is rendered. This is important when clicking the run button for the second time.
2. Incrementing an observable. This is a common gotcha. For immutable types (`int`, `str`, etc) `x += a` is the same as `x = x + a`. But for mutable types this is an in-place operation and Flet **may not** detect a change. ⚠️ Here it's fine since counter is float. See [appendix](/courses/app-dev/02-flet.html#appendix-flet-observables).
3. Here an instance of the `AppState` is used as state variable to trigger UI re-render with the reactive context (`ft.component`).
4. An observable parameter is assigned to the `ProgressBar`. 
5. Observables can define **methods** of modifying its internal attributes based on events, which we can utilize in event handlers.

:::{.callout-tip}
The distinction between **app state** (observables, "truth") and **app view** (UI components, "rendered") is a good mental model when designing declarative applications.

:::

<video
  src="./img/flet-declarative/counter.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

## Example: Declarative Routing

Note that the `page` object can be accessed using `ft.context.page`. In this example, we represent the app state as containing the route string and having a `go` method which triggers a re-assignment of the `route` observable parameter (hence getting a UI re-render), and pushing a new route (corresponds to URL in browser):

```python
import flet as ft
import asyncio
from dataclasses import dataclass

@ft.observable
@dataclass
class AppState:
    route: str = "/"

    async def go(self, route: str):
        self.route = route
        await ft.context.page.push_route(route)
```

Then, we define the page views as components which are all rendered separately:

```python
@ft.component
def HomePage(app: AppState) -> ft.Column:
    return ft.Column(
        controls=[
            ft.Text(f"Current route: {app.route}", size=24),
            ft.Button(
                "About",
                on_click=lambda e: app.go("/about"),
            ),
        ]
    )

@ft.component
def AboutPage(app: AppState) -> ft.Column:
    return ft.Column(
        controls=[
            ft.Text(f"Current route: {app.route}", size=24),
            ft.Button(
                "🏠",
                on_click=lambda e: app.go("/"),
            ),
        ]
    )

@ft.component
def NotFoundPage(app: AppState) -> ft.Column:
    return ft.Column(
        controls=[
            ft.Text(f"Current route: {app.route}", size=24),
            ft.Text("404 Not Found", size=24),
            ft.Button(
                "🏠",
                on_click=lambda e: app.go("/"),
            ),
        ]
    )
```

Finally, we define `AppView` for the UI. This relies on **defining** the `page.on_route_change` event handler as `app.go`: 

```python
@ft.component
def AppRoot():
    app, _ = ft.use_state(AppState())
    ft.context.page.on_route_change = lambda e: app.go(e.route)
    return Router(app)

@ft.component
def Router(app: AppState) -> ft.Control:
    match app.route:
        case "/":
            return HomePage(app)
        case "/about":
            return AboutPage(app)
        case _:
            return NotFoundPage(app)
```

**Remark.** Here the `Router` is just a wrapper around the defined page views. In more complex applications, the router handles logic such as [route templates](/courses/app-dev/01-flet.html#route-templates).

<video
  src="./img/flet-declarative/routing.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

## Appendix: Flet observables

In Flet’s declarative model, an `Observable` triggers UI updates when the **observable reference itself is reassigned**, not when the underlying object is mutated in place. Operations like `=` create a new value and rebind the observable, which Flet detects and use as a signal to re-render dependent components. Flet generally supports `list` and `dict` methods (e.g., `list.append()` or `dict["k"] = v` even `del dict["k"]` is supported). However, there are exceptions so you have to test your code (see below for a testing methodology).

:::{.callout-note}
For example, recall that the behavior of `+=` depends on the type: for **immutable types** (`int`, `str`, `tuple`), `+=` behaves like `=` and triggers a re-render, but for **mutable types** (`list`, `dict`, `set`), `+=` mutates the object in place. It turns out that this mutation does not trigger a notification. In this case, it's better to just reassign: `list = list + [item]`.

:::


### Assignment semantics

In general, this design mirrors many reactive frameworks where reactivity is tied to assignment semantics rather than deep mutation tracking, keeping the system simpler and more predictable, but requiring explicit reassignment when working with mutable structures. Moreover, mutation tracking is shallow. This limitation is somewhat expected since tracking of mutation in composite / nested objects does not scale well (exponential).

A design pattern for handling fields containing mutable objects is to assign an `int` ID:

```python
TaskID = ft.IdCounter() # <1>

@dataclass
class Item:
    id: int = field(default_factory=TaskID)

@ft.observable
@dataclass
class AppState:
    items: list[Item] = field(default_factory=list)
    
    def add_item(self, item: Item):
        self.items.append(item)
    
    def delete_item(self, item: Item):
        self.items = [t for t in self.items if t.id != item.id] # <2>
```
1. Flet provides a thread-safe unique ID generator.
2. Checking for object equality via the ID. The new list is reconstructed and assigned.

### Testing gotchas

For our tests we define nested observables and check mutations:

In [1]:
import flet as ft
from dataclasses import dataclass, field

@ft.observable
@dataclass
class Item:
    x: int
    y: int

@ft.observable
@dataclass
class AppState:
    items: list[Item] = field(default_factory=list)
    items_map: dict[int, Item] = field(default_factory=dict)


listener = lambda source, field: print(f"Change detected: {field} in {source}")

app = AppState()
app.subscribe(listener)

item = Item(1, 2)
item.subscribe(listener);

#### #1 Trigger on field *change*

Here we initialize two observables and subscribed with a listener to see whether a signal is being sent by the observable. Triggering the listener can be seen in the ff. example. Note that the signal is only emitted once since the field `x` only changed during the first time:

In [2]:
item.x = 10
item.x = 10

Change detected: x in Item(x=10, y=2)


#### #2 Supports list & dict methods

As mentioned above, Flet nicely implements observability nicely for `list` and `dict` methods:

In [3]:
app.items.append(item)
app.items_map[0] = item

Change detected: items in AppState(items=[Item(x=10, y=2)], items_map={})
Change detected: items_map in AppState(items=[Item(x=10, y=2)], items_map={0: Item(x=10, y=2)})


Some operations such as `+=` are unsupported:

In [4]:
app.items += [item]     # no notify for app... (；￣Д￣)

#### #3 When in doubt, re-assign

One can't go wrong with reassignment:

In [5]:
app.items = app.items + [item]

Change detected: items in AppState(items=[Item(x=10, y=2), Item(x=10, y=2), Item(x=10, y=2)], items_map={0: Item(x=10, y=2)})


#### #4 Only shallow changes are observed

Changing a nested observable (`item`) does not propagate to the main one (`app`):

In [6]:
# only the item listener is called, not app's
app.items[0].y = -1

Change detected: y in Item(x=10, y=-1)


### Using `notify`

To address the above limitations, we can manually **notify** the main app state each time a nested object is changed. Regarding the output below, we are only interested in notifications from `AppState`. This setup accomplishes precisely this:

In [7]:
def update_item(item_id: int, dx: int, dy: int):
    # NOTE: only update items using this method!
    if item_id in app.items_map:
        item = app.items_map[item_id]
        item.x = dx
        item.y = dy
        app.notify()        # !!

update_item(0, 5, 5)

Change detected: x in Item(x=5, y=-1)
Change detected: y in Item(x=5, y=5)
Change detected: None in AppState(items=[Item(x=5, y=5), Item(x=5, y=5), Item(x=5, y=5)], items_map={0: Item(x=5, y=5)})


:::{.callout-note}
The observable `.notify()` method makes the observable emit a signal without a change in any of its attributes. (You can see that in the printed message above with `None` as the field sent to the listener.) This mechanic ensures that messages are propagated upwards intentionally and by design. One consequence is that signals are emitted exponentially (unless there's some clever mechanism for pruning the updates).

:::

The downside is having to know where, or remembering, to write `.notify()`. From experience[^skill_issue], this can sometimes be forgotten since tracking places in the program to appropriately place the manual trigger may require nontrivial reasoning about the code. Otherwise, you may find yourself wondering why the UI is flickering, or why a component is desynced. In general, only notify to "commit" the application state (when everything about the state & data has been resolved).

[^skill_issue]: Skill issue? `¯\_(ツ)_/¯`

### Functional approach

An alternative approach is to have a [single flat observable]{.mark} to handle app state. This can be more scalable and easy to reason with, especially as the application grows. All other data models are dataclasses, and we maintain a [single observable]{.underline} for the entire application state.

In [8]:
import flet as ft
from dataclasses import dataclass, field

@dataclass      # NOTE: no @ft.observable here!
class Item:
    x: int
    y: int

@ft.observable
@dataclass
class AppState:
    items: list[Item] = field(default_factory=list)
    items_map: dict[int, Item] = field(default_factory=dict)

    def add_item(self, x: int, y: int) -> int:
        item_id = len(self.items)
        item = Item(x, y)
        self.items.append(item)
        self.items_map[item_id] = item
        return item_id

    def update_item(self, item_id: int, dx: int, dy: int):
        if item_id in self.items_map:
            self.items_map[item_id] = Item(dx, dy)    # NOTE: create a new item instance!

listener = lambda source, field: print(f"Change detected: {field} in {source}")
app = AppState()
app.subscribe(listener);

:::{.callout-note}
Observe that `self.notify()` is not used!

:::

Here we are getting two sets of notifications which is good:

In [9]:
item_id = app.add_item(1, 2)
app.update_item(item_id, 5, 5)

Change detected: items in AppState(items=[Item(x=1, y=2)], items_map={})
Change detected: items_map in AppState(items=[Item(x=1, y=2)], items_map={0: Item(x=1, y=2)})
Change detected: items_map in AppState(items=[Item(x=1, y=2)], items_map={0: Item(x=5, y=5)})


## Capstone: Declarative Todo App

In [4]:
import flet; print(flet.__version__)

0.80.1


In this section, we rewrite the [Todo App](/courses/app-dev/01-flet.html#capstone-todo-app) in the declarative style[^prev_imperative]. As in the above discussion, we will define observables and views that correspond to these. We reproduce the previous app with exactly the same features. The only difference is that we have a clear separation between state and UI. Hence, we need (1) [handle UI events]{.mark} when user interacts with the UI so that it interfaces to the correct state update method, and (2) ensure that the [UI is notified]{.mark} of changes in state via proper assignment mechanics, so that the re-render actually happens, with UI reflecting the current state. See image below (@fig-declarative-model).

[^prev_imperative]: Our previous imperative code explicitly modifies controls and their values to reflect the true state.

![**Declarative separation.** Our code is analogous to this diagram where application state and UI are separated by an interface. It's important to note that event handlers in the UI interface to call the appropriate state update methods. In particular, [event handlers must not modify the components explicitly]{.mark}. Recall that declarative code describes the view based on the state, so it suffices for event handlers to modify the state which is reflected in the UI. In other words, $\text{UI}^\prime = f( \text{State}^\prime).$ [Source](https://docs.flet.dev/cookbook/declarative-vs-imperative-crud-app/#example-2-declarative)](./img/flet-declarative/crud-declarative.drawio.png){#fig-declarative-model}

:::{.callout-note}
The caption in @fig-declarative-model is very important and describes the main idea behind the declarative approach to UI programming. Declarative programming work is divided into two parts: (1) describing the UI in terms of the state (i.e. components as a function of state variables[^declarative_eqn]), (2) ensuring that UI event handlers are ultimately tied to modifying [first-order]{.underline} state variables, or fields, that notify the observable (e.g. not modifying derived attributes). This suffices to get a consistent UI as the framework (in our case, Flet) handles the reactive process.

:::

[^declarative_eqn]: $\text{UI} = f(\text{State})$

### Task item

Our task item is simply a dataclass (i.e. not an observable):

```python
import flet as ft

from typing import Callable, Optional
from dataclasses import dataclass, field

TaskID = ft.IdCounter()

@dataclass
class Task:
    name: str
    is_completed: bool = False
    id: int = field(default_factory=TaskID)
```

**NOTE:** This still encodes state even if its not an observable. Its fields are just not used to trigger a re-render. Instead, updates to fields of `Task` instances have to be tracked within an encompassing observable.

### App state

For this example, we maintain a single observable for the application state[^single_observable_approach]. The app state is largely described by *two* variables, the task list (containing `Task` objects that each encode task status), and the index of the tab filter. So we have precisely that as fields. Next, the class defines methods for adding, updating, filtering (via switching the tab filter), and deleting tasks. Note that all of these follow re-assignment to signal a UI re-render.

[^single_observable_approach]: See [appendix](http://localhost:4200/courses/app-dev/02-flet.html#functional-approach) for a discussion of this approach.

```python
ALL = "all"
ACTIVE = "active"
COMPLETED = "completed"

@ft.observable
@dataclass
class TodoAppState:
    tasks: list[Task] = field(default_factory=list)
    task_filters: list[str] = field(default_factory=lambda: [ALL, ACTIVE, COMPLETED])
    selected_filter_idx: int = 0

    def add_task(self, task: Task):
        self.tasks.append(task)
    
    def delete_task(self, task: Task):
        self.tasks = [t for t in self.tasks if t.id != task.id]

    def update_task(self, task: Task, new_name: str):
        for idx, t in enumerate(self.tasks):
            if t.id == task.id:
                break
        self.tasks[idx] = Task(
            name=new_name, 
            is_completed=t.is_completed, 
            id=t.id
        )
    
    def toggle_task_status(self, task: Task):
        for idx, t in enumerate(self.tasks):
            if t.id == task.id:
                break
        self.tasks[idx] = Task(
            name=t.name, 
            is_completed=not t.is_completed, 
            id=t.id
        )

    def switch_filter(self, filter_idx: int):
        self.selected_filter_idx = filter_idx

    @property
    def visible_tasks(self) -> list[Task]:
        tab = self.task_filters[self.selected_filter_idx]
        is_visible = {
            ALL: lambda task: True,
            COMPLETED: lambda task: task.is_completed,
            ACTIVE: lambda task: not task.is_completed
        }
        return [t for t in self.tasks if is_visible[tab](t)]

    @property
    def active_tasks_number(self) -> int:
        return len([task for task in self.tasks if not task.is_completed])
```

### Task view

We now proceed with UI components. Recall that we previously had to hide parts of the UI like the edit field when the text is in display mode. Here, 
we don't need to set the visibility of the controls, we simply do **conditional rendering**. That is, we have conditionals on what controls get returned by the component function depending on the state.

Observe that `TaskView` takes in `app: TodoAppState` as input. This allows the view to communicate with the larger app with regards to the events that happen in this component (e.g. toggle status, deletion) that affects the main app (# visible tasks).

```python
@ft.component
def TaskView(app: TodoAppState, task: Task) -> ft.Row:

    is_editing, set_is_editing = ft.use_state(False)    # <1>
    _name, set_name = ft.use_state(task.name)

    def start_edit():
        set_name(task.name)
        set_is_editing(True)

    def cancel_edit():
        set_is_editing(False)

    def save_edit():
        app.update_task(task, new_name=_name)
        set_is_editing(False)

    def confirm_delete():
        dialog = ConfirmDialog(
            text="Are you sure you want to delete this task?",
            confirm_handler=lambda e: app.delete_task(task)
        )
        ft.context.page.show_dialog(dialog)

    if is_editing:
        return ft.Row([
            ft.TextField(
                value=_name,
                expand=True, 
                on_change=lambda e: set_name(e.control.value),  # <2>
                on_submit=save_edit,
                autofocus=True
            ),
            ft.IconButton(
                icon=ft.Icons.SAVE,
                on_click=save_edit
            ),
            ft.IconButton(
                icon=ft.Icons.STOP,
                on_click=cancel_edit
            )
        ])
    else:
        return ft.Row([
            ft.Checkbox(
                value=task.is_completed,
                label=task.name,
                on_change=lambda e: app.toggle_task_status(task)    # <3>
            ), 
            ft.IconButton(
                icon=ft.Icons.EDIT, 
                on_click=start_edit
            ),
            ft.IconButton(
                icon=ft.Icons.DELETE, 
                on_click=confirm_delete # <4>
            )
        ])
```

1. Here we define a flag `is_editing` which is a Boolean for whether we're in edit mode or not. And we have `_name` which is a variable the name temporary value in the text field when editing. Next, in the return part we see two row controls being returned depending on `is_editing`. For edit mode, we have text field, save, and cancel. For display mode, we have checkbox, edit, and delete. These have corresponding event handlers. 

2. In the `TextField` event handler, the `_name` parameter stores the value in the text field for each keystroke. The final value is persisted on submission, i.e. when `save_edit` is called to run `app.update_task(new_name=_name)` updating the task name.

3. Because the checkbox state is declaratively bound to the task attributes, the UI consistently reflects the truth in the data model. Clicking the checkbox invokes `toggle_task_status` of the `app` observable that serves to update the underlying `task.is_completed` attribute and trigger a rerender. 

4. Delete opens a **dialog modal** which shows a confirmation prompt:

```python
# NOTE: non-reactive => suffices to have this as usual fn that returns a control
# If it needs things like ft.use_state, then it should be wrapped as @ft.component 
# to support a reactive context.
def ConfirmDialog(
    text: Optional[str] = "",
    decline_handler: Optional[Callable] = None,
    confirm_handler: Optional[Callable] = None,
):
    def wrap_close(handler: Optional[Callable]):
        if handler is None:
            return lambda e: e.page.pop_dialog()
        return lambda e: (handler(e), e.page.pop_dialog())

    return ft.AlertDialog(
        modal=True,
        title=ft.Text("Confirm delete"),
        content=ft.Text(text),
        actions=[
            ft.Button("Yes", on_click=wrap_close(confirm_handler)),
            ft.TextButton("No", on_click=wrap_close(decline_handler)),
        ],
        actions_alignment=ft.MainAxisAlignment.END
    )
```

:::{.callout-tip}
Note that the handlers only use methods defined in the observable class (e.g. `app.task_update` or `app.toggle_task_status`). No explicit modification of Flet controls are done, ensuring everything is consistent with the state. See @fig-declarative-model. The event handlers also use the setters to set as appropriate the `is_editing` flag depending on the UI event.

:::

### App view

Similar to task views the app view implements event handlers that process user interaction in the UI and wraps the appropriate state methods.
To understand this (in general, to read component functions), we start with the returned controls, and work backwards on local `use_state` variables and
the control event handlers.

```python
@ft.component
def TodoAppView() -> ft.Column:
    todo, _ = ft.use_state(TodoAppState())

    # add new task
    new_task_name, set_new_task_name = ft.use_state("")
    new_task_field_ref = ft.use_ref()

    async def add_task():
        task = Task(name=new_task_name, is_completed=False)
        todo.add_task(task)
        set_new_task_name("")
        await new_task_field_ref.current.focus()  # refocus after adding task

    new_task_field = ft.Row(controls=[  # <2>
        ft.TextField(
            ref=new_task_field_ref,
            hint_text="What needs to be done?",
            value=new_task_name,
            expand=True, 
            on_submit=add_task,
            on_change=lambda e: set_new_task_name(e.control.value),
            autofocus=True   # only works on first render
        ),
        ft.FloatingActionButton(
            icon=ft.Icons.ADD, 
            on_click=add_task
        )
    ])

    # status filter tabs
    filter_tabs = ft.Tabs(
        selected_index=todo.selected_filter_idx,    # <3>
        length=3,
        on_change=lambda e: todo.switch_filter(e.control.selected_index),
        content=ft.TabBar(
            scrollable=False,
            tabs=[ft.Tab(label=tab) for tab in todo.task_filters],
        )
    )

    # footer
    def delete_completed():
        for task in todo.tasks[:]:
            if task.is_completed:
                todo.delete_task(task)

    def confirm_delete_completed():
        dialog = ConfirmDialog(
            text="Are you sure you want to delete completed tasks?",
            confirm_handler=lambda e: delete_completed()
        )
        ft.context.page.show_dialog(dialog)

    # build ui
    return ft.Column(
        horizontal_alignment=ft.CrossAxisAlignment.CENTER,
        controls=[
            Header(),   # <1>
            new_task_field, # <2>
            filter_tabs,
            ft.ListView(
                controls=[
                    TaskView(app=todo, task=t)  # <4>
                    for t in todo.visible_tasks
                ],
                height=250, 
                spacing=10, 
                scroll=ft.ScrollMode.ALWAYS,
                auto_scroll=True,
            ),
            ft.Divider(thickness=0.5, color=ft.Colors.GREY_600),
            Footer(todo.active_tasks_number, confirm_delete_completed)  # <5>
        ],
    )

@ft.component
def Header():
    return ft.Text("Todo list 📝", size=50, weight=ft.FontWeight.BOLD)

@ft.component
def Footer(count_active_tasks: int, delete_completed_handler: Callable):
    return ft.Row(
        [
            ft.Text(f"{count_active_tasks} active tasks left.", color=ft.Colors.GREY_400),
            ft.Button(
                "Clear Completed", 
                on_click=delete_completed_handler,
                style=ft.ButtonStyle(shape=ft.RoundedRectangleBorder(radius=10))
            )
        ],
        alignment=ft.MainAxisAlignment.SPACE_BETWEEN
    )


def main(page: ft.Page):
    page.render(TodoAppView)


if __name__ == "__main__":
    ft.run(main)

```

1. The application view has a header (which is simply `Text`). This is followed by the text field for adding new tasks. Then, we have task tab filters for showing tasks with different status. Selecting the filter determines which task are listed. Finally, the footer shows the number of active tasks.

2. Note that this uses a `ft.use_ref()` which gives you a **stable handle** to a UI control instance across rerenders so you can imperatively interact with it. Flet recreates controls each render, hence invalidating references. Here we can do `.current` to get the latest rendered control instance. By assigning this to the `ref` field of the new task field, we are able to refocus the text field each time we create a task via `.focus()` on the returned control instance in the `add_task` event handler. Note `set_new_task_name(e.control.value)` here as in the edit field for task items, that updates every keystroke. The final value is committed in `add_task` upon submission.

3. The tab filter UI is consistent with the app state variable `selected_filter_idx`. This is updated via the `use_state` setter every time we select a new tab in the UI ensuring consistency between local state and app state.

4. Task views are constructed at each render based on the app state. Again, grounding on the app state makes everything consistent. And only using `use_state` setters for transferring local data between components and re-renders. 

5. Number of active tasks is computed from the state at each render. This is consistent with the task views list, since its value is based on the app state object. Finally, the footer has a button for deleting the completed tasks for improving UX.

## Capstone: Chatroom App

In [3]:
import flet; print(flet.__version__)

0.80.1


We build a multi-user **Chatroom App** with [Flet](https://flet.dev). The app is developed in three iterations. `v1` establishes the core: broadcasting messages across sessions using Flet's built-in **PubSub library**. `v2` adds usernames and shared in-memory state, introducing thread locking to handle concurrent access. `v3` polishes the UI with a custom `ChatMessage` control, circle avatars, autofocus, and autoscroll. Together these iterations cover the essential patterns for building interactive, multi-session Flet apps.

### Broadcasting with PubSub

<video
  src="./img/flet-chat/v1.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

Run this using `flet run --web v1.py` and open the same link in separate browsers creating two sessions.

```python
import flet as ft
from dataclasses import dataclass

@dataclass
class Message:
    user: str
    text: str

@ft.component
def AppView():
    page = ft.context.page
    session_id = page.session.id
    history, set_history = ft.use_state([])     # <1>
    message, set_message = ft.use_state("")
    
    def on_message(msg_obj: Message):           # <3>
        page.run_thread(lambda: set_history(lambda h: [*h, msg_obj]))

    # subscribe once. use_effect expects cleanup function
    def subscribe():                                        
        page.pubsub.subscribe(on_message)
        def cleanup(): 
            page.pubsub.unsubscribe(on_message)
        return cleanup

    # empty deps => run once on mount, and cleanup on unmount
    ft.use_effect(subscribe, [])        # <2>

    def send_click(e):  # <4>
        page.pubsub.send_all(msg_obj=Message(user=session_id, text=message))
        set_message("")

    return ft.Column(
        controls=[
            ft.Column(controls=[ft.Text(f"{m.user}: {m.text}") for m in history]),
            ft.Row(controls=[
                ft.TextField(
                    label="New message",
                    value=message,
                    width=400,
                    on_change=lambda e: set_message(e.control.value),
                    on_submit=send_click
                ),
                ft.Button("Send", on_click=send_click)
            ]),
        ]
    )


if __name__ == "__main__":
    ft.run(lambda page: page.render(AppView))
```

1. `history` and `message` are initialized with `use_state` so they persist across re-renders.

2. The effect `subscribe` runs only once at mount because the dependency list is empty (`[]`). See the docs on [`use_effect`](https://docs.flet.dev/types/useeffect/?h=use_effect). It returns a cleanup function that unsubscribes from PubSub. The subscription causes a FastAPI worker to invoke `on_message` whenever a message is published; closure over `page` is what makes this work.

3. `set_history` is called with an *updater function* rather than a plain value, ensuring the callback always sees the latest state. The page auto-updates after the `run_thread` callback completes — no explicit `page.update()` needed[^run_thread].

4. Sending is equally straightforward: `page.pubsub.send_all` accepts any Python object and delivers it as the argument to every subscriber's `on_message` callback (hence the `msg_obj: Message` signature). After sending, `message` is cleared.

[^run_thread]: The exact mechanism is unclear, but this is what reliably runs everything in the correct render context without UX issues or error logs.

### Adding usernames

Users must choose a unique username when they join the chatroom. We track active users in an `active_users` set. Because multiple users connect concurrently, shared state requires careful coordination to stay consistent. In particular, we need to guard against **race conditions**: for example, two users could both pass the uniqueness check for the same name before either has been added to the set, resulting in duplicate usernames.

To prevent this, we use a **thread lock** to ensure that only one thread accesses `active_users` at a time.

```python
@dataclass
class ChatRoom:
    active_users: set[str] = field(default_factory=set)
    _lock: threading.Lock = field(default_factory=threading.Lock, repr=False)

    def add_user(self, name: str):
        with self._lock:
            self.validate_username(name)
            self.active_users.add(name)

    def remove_user(self, name: str):
        with self._lock:
            self.active_users.discard(name)

    def validate_username(self, name: str):
        if name in self.active_users:
            raise ValueError(f'"{name}" is already taken. Please choose another.')
        if not name.strip():
            raise ValueError("Username cannot be empty.")
```

Chatroom [state is shared]{.mark} across sessions by attaching a single `ChatRoom` instance to every page:

```python
if __name__ == "__main__":
    # shared state across sessions
    chatroom = ChatRoom()

    def bootstrap(page: ft.Page):
        page.chat = chatroom
        page.render(AppView)

    ft.run(bootstrap)
```

Next, we have the **join dialog**. To understand this, let's walk backwards:

```python
def JoinDialog(join_click: Callable, chatroom: ChatRoom):
    def join_click_loop():                                                          # <1>
        def handler(e):
            try:
                e.page.pop_dialog()
                entered = username.value.strip()
                chatroom.add_user(entered)
                join_click(e, entered)                                              # <2>

            except ValueError as error:                                             # <3>
                e.page.pop_dialog()
                e.page.show_dialog(
                    ft.AlertDialog(
                        modal=True,
                        title=ft.Text("Invalid Username"),
                        content=ft.Text(str(error)),
                        actions=[
                            ft.Button(
                                "OK", 
                                on_click=lambda _: (                    
                                    e.page.pop_dialog(),                                    # pop error dialog        
                                    e.page.show_dialog(JoinDialog(join_click, chatroom))    # start over with a fresh join dialog
                                )
                            )
                        ],
                        actions_alignment=ft.MainAxisAlignment.END,
                    )
                )
                return
        return handler
    
    username = ft.TextField(
        label="Enter your name",
        on_submit=join_click_loop(),
        autofocus=True,
    )

    return ft.AlertDialog(
        modal=True, 
        title=ft.Text("Welcome!"),
        content=ft.Column([username], tight=True),
        actions=[ft.Button("Join", on_click=join_click_loop())], 
        actions_alignment=ft.MainAxisAlignment.END
    )
```
1. `join_click_loop` builds a handler that retries until a valid username is submitted — it keeps opening and closing dialogs until the happy path is reached. Each entry into the loop begins by dismissing the current modal with `pop_dialog`. Both pressing ENTER and clicking "Join" invoke this loop.
2. The happy path calls `join_click(e, entered)` with the validated username.
3. If the username is already taken or empty, a `ValueError` is raised. The join dialog is closed, an error dialog is shown, and the loop restarts. Because the error dialog is non-blocking, the handler must `return` immediately after showing it to avoid continuing execution.

Since a join dialog is opened at start up, we upgrade `subscribe` to `join_and_subscribe`:

```python
    def join_and_subscribe():
        page.pubsub.subscribe(on_message)

        try:
            stored_username = page.session.store.get("username") or ""
            page.chat.add_user(stored_username)
            set_username(stored_username)
            
        except ValueError:
            def on_join(e, entered_name):
                set_username(entered_name)
                page.session.store.set("username", entered_name)
                page.pubsub.send_all(
                    Message(
                        user="System", 
                        text=f"{entered_name} joined the chat!"
                    )
                )

            page.show_dialog(JoinDialog(on_join, page.chat))

        def cleanup():
            current_user = page.session.store.get("username") or ""
            page.chat.remove_user(current_user)
            page.pubsub.unsubscribe(on_message)

        return cleanup

    ft.use_effect(join_and_subscribe, [])
```

<video
  src="./img/flet-chat/v2.mp4"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  onloadeddata="this.playbackRate=1.25"
  style="max-width:100%;">
</video>

### UI enhancements

#### Chat messages

When a user joins, we print a **system message** that a user has joined in italicized gray. System messages (e.g. join notifications) are rendered as italicized gray text. Regular user messages use a custom `ChatMessage` control with a [circle avatar]{.underline}:

```{.python filename="src/v3.py"}
@ft.control
class ChatMessage(ft.Row):
    def __init__(self, message: Message):
        super().__init__()
        self.message = message
        self.vertical_alignment = ft.CrossAxisAlignment.START
        self.controls = [
            ft.CircleAvatar(
                content=ft.Text(self.get_initials(self.message.user)),
                color=ft.Colors.WHITE,
                bgcolor=self.get_avatar_color(self.message.user),
            ),
            ft.Column(
                tight=True,
                spacing=5,
                controls=[
                    ft.Text(self.message.user, weight=ft.FontWeight.BOLD),
                    ft.Text(self.message.text, selectable=True),
                ],
            ),
        ]

    def get_initials(self, username: str):
        if username:
            return username[:1].capitalize()
        else:
            return "?"

    def get_avatar_color(self, username: str):
        colors_lookup = [
            ft.Colors.AMBER,
            ft.Colors.BLUE,
            ft.Colors.BROWN,
            ft.Colors.CYAN,
            ft.Colors.GREEN,
            ft.Colors.INDIGO,
            ft.Colors.LIME,
            ft.Colors.ORANGE,
            ft.Colors.PINK,
            ft.Colors.PURPLE,
            ft.Colors.RED,
            ft.Colors.TEAL,
            ft.Colors.YELLOW,
        ]
        color_idx = hash(username) % len(colors_lookup)
        return colors_lookup[color_idx]


def build_messages(messages: list[Message]) -> list[ft.Control]:
    controls = []
    for msg in messages:
        if msg.user == "System":
            text = msg.text
            system_message = ft.Text(text, italic=True, color=ft.Colors.GREY)
            controls.append(system_message)
        else:
            controls.append(ChatMessage(msg))
    return controls
```

:::{.callout-caution}
Setting `color=ft.colors.GREY` or `color=ft.Colors.GRAY` by mistake silently fails, i.e. the messages are not shown in the UI but no exception is raised! IDE autocomplete and syntax highlighting are crucial to ensure that we are using an existing object.

:::

#### Arranging the chat panel

The page is a column consisting of a **container** (chat history) and a **row** (new message input). The container holds a `ListView` for scrollable content, where each item is either a `ChatMessage` control (regular messages) or a plain `Text` (system messages). The input row contains a `TextField` and a send button. `expand=True` is used throughout to fill available space proportionally.

![**Anatomy of our chat app UI.** It's good practice to draw a design like this by hand before coding.](./img/flet-chat/page-layout.svg)

`send_click` is made async so we can `await new_message.focus()` after sending, returning the cursor to the input field automatically. `autofocus=True` is also set on the `TextField` and the join dialog's username field so the keyboard is ready immediately. Otherwise, users would need to click the field before every message.

Finally, the chat history is wrapped in a `Container` for a bordered layout. Using `ListView` with `auto_scroll=True` means the view scrolls to the latest message whenever new content overflows the container. Both of these are important UX details. `expand=True` is used throughout to fill available space.

```{.python filename="src/v3.py"}
...
@ft.component
def AppView():
    ...

    async def send_click(e):
        if message.strip() and username:
            page.pubsub.send_all(Message(user=username, text=message))
            set_message("")
            await new_message.focus()

    chat = ft.ListView(
        controls=build_messages(history),
        expand=True,
        spacing=10,
        auto_scroll=True,
    )

    new_message = ft.TextField(
        label="New message",
        value=message,
        expand=True,
        on_change=lambda e: set_message(e.control.value),
        on_submit=send_click,
        autofocus=True,
    )

    new_message_row = ft.Row(
        controls=[
            new_message,
            ft.Button(content=ft.Icon(ft.Icons.SEND), on_click=send_click)
        ],
    )

    return ft.Column(
        controls=[
            ft.Container(
                content=chat,
                border=ft.Border.all(1, ft.Colors.OUTLINE),
                border_radius=5,
                padding=10,
                expand=True,
            ),
            new_message_row
        ],
        expand=True,
    )
```

<br>

**Multi-user demo.** Layout, autoscroll, and autofocus are emphasized in `v3`:

<video
  src="./img/flet-chat/v3.mp4"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  onloadeddata="this.playbackRate=1.5"
  style="max-width:100%;">
</video>

<br>

**Source**

In [ ]:
#| code-fold: true
import flet as ft
import threading

from typing import Callable, Optional
from dataclasses import dataclass, field


@dataclass
class ChatRoom:
    active_users: set[str] = field(default_factory=set)
    _lock: threading.Lock = field(default_factory=threading.Lock, repr=False)

    def add_user(self, name: str):
        with self._lock:
            self.validate_username(name)
            self.active_users.add(name)

    def remove_user(self, name: str):
        with self._lock:
            self.active_users.discard(name)

    def validate_username(self, name: str):
        if name in self.active_users:
            raise ValueError(f'"{name}" is already taken. Please choose another.')
        if not name.strip():
            raise ValueError("Username cannot be empty.")


def JoinDialog(join_click: Callable, chatroom: ChatRoom):
    def join_click_loop():                                                          # <1>
        def handler(e):
            try:
                e.page.pop_dialog()
                entered = username.value.strip()
                chatroom.add_user(entered)
                join_click(e, entered)                                              # <2>

            except ValueError as error:                                             # <3>
                e.page.pop_dialog()
                e.page.show_dialog(
                    ft.AlertDialog(
                        modal=True,
                        title=ft.Text("Invalid Username"),
                        content=ft.Text(str(error)),
                        actions=[
                            ft.Button(
                                "OK", 
                                on_click=lambda _: (                    
                                    e.page.pop_dialog(),                                    # pop error dialog        
                                    e.page.show_dialog(JoinDialog(join_click, chatroom))    # start over with a fresh join dialog
                                )
                            )
                        ],
                        actions_alignment=ft.MainAxisAlignment.END,
                    )
                )
                return
        return handler
    
    username = ft.TextField(
        label="Enter your name",
        on_submit=join_click_loop(),
        autofocus=True,
    )

    return ft.AlertDialog(
        modal=True, 
        title=ft.Text("Welcome!"),
        content=ft.Column([username], tight=True),
        actions=[ft.Button("Join", on_click=join_click_loop())], 
        actions_alignment=ft.MainAxisAlignment.END
    )


@dataclass
class Message:
    user: str
    text: str


@ft.control
class ChatMessage(ft.Row):
    def __init__(self, message: Message):
        super().__init__()
        self.message = message
        self.vertical_alignment = ft.CrossAxisAlignment.START
        self.controls = [
            ft.CircleAvatar(
                content=ft.Text(self.get_initials(self.message.user)),
                color=ft.Colors.WHITE,
                bgcolor=self.get_avatar_color(self.message.user),
            ),
            ft.Column(
                tight=True,
                spacing=5,
                controls=[
                    ft.Text(self.message.user, weight=ft.FontWeight.BOLD),
                    ft.Text(self.message.text, selectable=True),
                ],
            ),
        ]

    def get_initials(self, username: str):
        if username:
            return username[:1].capitalize()
        else:
            return "?"

    def get_avatar_color(self, username: str):
        colors_lookup = [
            ft.Colors.AMBER,
            ft.Colors.BLUE,
            ft.Colors.BROWN,
            ft.Colors.CYAN,
            ft.Colors.GREEN,
            ft.Colors.INDIGO,
            ft.Colors.LIME,
            ft.Colors.ORANGE,
            ft.Colors.PINK,
            ft.Colors.PURPLE,
            ft.Colors.RED,
            ft.Colors.TEAL,
            ft.Colors.YELLOW,
        ]
        color_idx = hash(username) % len(colors_lookup)
        return colors_lookup[color_idx]


def build_messages(messages: list[Message]) -> list[ft.Control]:
    controls = []
    for msg in messages:
        if msg.user == "System":
            text = msg.text
            system_message = ft.Text(text, italic=True, color=ft.Colors.GREY)
            controls.append(system_message)
        else:
            controls.append(ChatMessage(msg))
    return controls


@ft.component
def AppView():
    page = ft.context.page
    username, set_username = ft.use_state("")
    history, set_history = ft.use_state([])
    message, set_message = ft.use_state("")

    def on_message(msg_obj: Message):
        page.run_thread(lambda: set_history(lambda h: [*h, msg_obj]))

    def join_and_subscribe():
        page.pubsub.subscribe(on_message)

        try:
            stored_username = page.session.store.get("username") or ""
            page.chat.add_user(stored_username)
            set_username(stored_username)
            
        except ValueError:
            def on_join(e, entered_name):
                set_username(entered_name)
                page.session.store.set("username", entered_name)
                page.pubsub.send_all(
                    Message(
                        user="System", 
                        text=f"{entered_name} joined the chat!"
                    )
                )

            page.show_dialog(JoinDialog(on_join, page.chat))

        def cleanup():
            current_user = page.session.store.get("username") or ""
            page.chat.remove_user(current_user)
            page.pubsub.unsubscribe(on_message)

        return cleanup

    ft.use_effect(join_and_subscribe, [])

    async def send_click(e):
        if message.strip() and username:
            page.pubsub.send_all(Message(user=username, text=message))
            set_message("")
            await new_message.focus()

    chat = ft.ListView(
        controls=build_messages(history),
        expand=True,
        spacing=10,
        auto_scroll=True,
    )

    new_message = ft.TextField(
        label="New message",
        value=message,
        expand=True,
        on_change=lambda e: set_message(e.control.value),
        on_submit=send_click,
        autofocus=True,
    )

    new_message_row = ft.Row(
        controls=[
            new_message,
            ft.Button(content=ft.Icon(ft.Icons.SEND), on_click=send_click)
        ],
    )

    return ft.Column(
        controls=[
            ft.Container(
                content=chat,
                border=ft.Border.all(1, ft.Colors.OUTLINE),
                border_radius=5,
                padding=10,
                expand=True,
            ),
            new_message_row
        ],
        expand=True,
    )


if __name__ == "__main__":
    # shared state across sessions
    chatroom = ChatRoom()

    def bootstrap(page: ft.Page):
        page.chat = chatroom
        page.render(AppView)

    ft.run(bootstrap)